# koELECTRA 학습 노트북 (Coupang v1.1)

이 노트북은 의도 분류를 위해 `monologg/koelectra-base-v3-discriminator` 모델을 파인튜닝합니다:
- `coupang/func_data/coupang_v1.1/train.csv`
- `coupang/func_data/coupang_v1.1/test.csv`
- `coupang/func_data/coupang_v1.1/label_map.csv`

범위: 학습 + 평가 + 아티팩트 저장 + 샘플 추론

In [11]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

2.10.0+cu126
True
NVIDIA GeForce RTX 4070 Laptop GPU


In [ ]:
!nvidia-smi

Fri Feb 13 14:59:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P8              2W /   28W |     852MiB /   8188MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

: 

In [12]:
# 셀 1: 환경/버전 확인
# 라이브러리 import, 시드 고정, CUDA/GPU 상태를 확인합니다.

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import evaluate
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'torch: {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'datasets: {datasets.__version__}')
print(f'evaluate: {evaluate.__version__}')
print(f'cuda_available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu_name: {torch.cuda.get_device_name(0)}')


torch: 2.10.0+cu126
transformers: 5.1.0
datasets: 4.5.0
evaluate: 0.4.6
cuda_available: True
gpu_name: NVIDIA GeForce RTX 4070 Laptop GPU


In [ ]:
# 셀 2: 경로/하이퍼파라미터 설정
# 데이터 경로, 모델명, 출력 경로와 학습 설정값을 집중 관리합니다.

DATA_DIR = Path('coupang/func_data/coupang_v1.1')
MODEL_NAME = 'monologg/koelectra-base-v3-discriminator'
OUTPUT_DIR = Path('model_training/outputs/koelectra_coupang_v1_1')

MAX_LENGTH = 64
TRAIN_BS = 8
EVAL_BS = 16
GRAD_ACCUM = 4
LR = 2e-5
EPOCHS = 6
FP16 = True
EARLY_STOP = 2
VAL_SIZE = 0.1

if not DATA_DIR.exists():
    alt_data_dir = Path('../coupang/func_data/coupang_v1.1')
    if alt_data_dir.exists():
        DATA_DIR = alt_data_dir

if not OUTPUT_DIR.parent.exists():
    alt_output_dir = Path('../model_training/outputs/koelectra_coupang_v1_1')
    if alt_output_dir.parent.exists() or Path('../model_training').exists():
        OUTPUT_DIR = alt_output_dir

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR  :', DATA_DIR.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())


DATA_DIR  : C:\ssafy\공통설계\coupang\func_data\coupang_v1.1
OUTPUT_DIR: C:\ssafy\공통설계\model_training\outputs\koelectra_coupang_v1_1


In [14]:
# 셀 3: 데이터 로드/검증 + 라벨 매핑
# CSV 필수 컬럼을 검사하고 label_id를 class_idx로 변환합니다.

required_cols = {'text', 'label_id', 'label'}
label_map_required_cols = {'label_id', 'label'}

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
label_map_df = pd.read_csv(DATA_DIR / 'label_map.csv')

for name, df in {'train': train_df, 'test': test_df}.items():
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{name}.csv missing required columns: {sorted(missing)}')

missing = label_map_required_cols - set(label_map_df.columns)
if missing:
    raise ValueError(f'label_map.csv missing required columns: {sorted(missing)}')

train_df['label_id'] = train_df['label_id'].astype(int)
test_df['label_id'] = test_df['label_id'].astype(int)
label_map_df['label_id'] = label_map_df['label_id'].astype(int)

label_map_df = label_map_df.sort_values('label_id').reset_index(drop=True)
label_map_df['class_idx'] = range(len(label_map_df))

label_id_to_class_idx = dict(zip(label_map_df['label_id'], label_map_df['class_idx']))
class_idx_to_label = dict(zip(label_map_df['class_idx'], label_map_df['label']))
label_to_class_idx = dict(zip(label_map_df['label'], label_map_df['class_idx']))

unknown_train_label_ids = sorted(set(train_df['label_id']) - set(label_id_to_class_idx))
unknown_test_label_ids = sorted(set(test_df['label_id']) - set(label_id_to_class_idx))
if unknown_train_label_ids:
    raise ValueError(f'train.csv contains unknown label_id: {unknown_train_label_ids}')
if unknown_test_label_ids:
    raise ValueError(f'test.csv contains unknown label_id: {unknown_test_label_ids}')

train_df['class_idx'] = train_df['label_id'].map(label_id_to_class_idx)
test_df['class_idx'] = test_df['label_id'].map(label_id_to_class_idx)

num_classes = len(label_map_df)
if sorted(label_map_df['class_idx'].tolist()) != list(range(num_classes)):
    raise ValueError('class_idx mapping must be contiguous from 0 to num_classes-1')

print(f'train rows: {len(train_df)}')
print(f'test rows : {len(test_df)}')
print(f'num classes: {num_classes}')

train_dist = train_df['class_idx'].value_counts().sort_index().rename('train_count')
test_dist = test_df['class_idx'].value_counts().sort_index().rename('test_count')
dist_df = pd.concat([train_dist, test_dist], axis=1).fillna(0).astype(int)

display(label_map_df[['label_id', 'label', 'class_idx']].head())
display(dist_df.head(10))


train rows: 6840
test rows : 1710
num classes: 28


,label_id,label,class_idx
0,0,unknown,0
1,68,go_hearbe,1
2,69,search_products,2
3,70,click_order_history,3
4,71,click_cart,4


,train_count,test_count
class_idx,,
0,360,90
1,240,60
2,240,60
3,240,60
4,240,60
5,240,60
6,240,60
7,240,60
8,240,60


In [15]:
# 셀 4: 전처리/데이터셋 변환
# train/val 분할(stratified)후 토크나이징하여 HF Dataset(torch)으로 만듭니다.

train_split_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df['class_idx'],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    base = df[['text', 'class_idx']].rename(columns={'class_idx': 'labels'}).copy()
    return Dataset.from_pandas(base, preserve_index=False)

hf_train = to_hf_dataset(train_split_df)
hf_val = to_hf_dataset(val_df)
hf_test = to_hf_dataset(test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

tokenized_train = hf_train.map(tokenize_batch, batched=True)
tokenized_val = hf_val.map(tokenize_batch, batched=True)
tokenized_test = hf_test.map(tokenize_batch, batched=True)

columns_to_keep = ['input_ids', 'attention_mask', 'labels']
if 'token_type_ids' in tokenized_train.column_names:
    columns_to_keep.append('token_type_ids')

tokenized_train.set_format(type='torch', columns=columns_to_keep)
tokenized_val.set_format(type='torch', columns=columns_to_keep)
tokenized_test.set_format(type='torch', columns=columns_to_keep)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print(f'train split: {len(tokenized_train)}')
print(f'val split  : {len(tokenized_val)}')
print(f'test split : {len(tokenized_test)}')


Map:   0%|          | 0/6156 [00:00<?, ? examples/s]

Map:   0%|          | 0/684 [00:00<?, ? examples/s]

Map:   0%|          | 0/1710 [00:00<?, ? examples/s]

train split: 6156
val split  : 684
test split : 1710


In [16]:
# 셀 5: 모델/Trainer 설정
# koELECTRA 분류 모델, TrainingArguments, 평가지표(accuracy/F1)를 설정합니다.

id2label = {int(idx): label for idx, label in class_idx_to_label.items()}
label2id = {label: int(idx) for idx, label in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_macro_f1',
    greater_is_better=True,
    fp16=FP16 and torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP)],
)

print('Trainer is ready.')


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missin

Trainer is ready.


In [17]:
# 셀 6: 학습 실행
# Trainer.train()을 실행하고 best checkpoint와 val 지표를 확인합니다.

train_result = trainer.train()
trainer.save_state()

print('Training finished.')
print('Best checkpoint:', trainer.state.best_model_checkpoint)

val_metrics = trainer.evaluate(eval_dataset=tokenized_val, metric_key_prefix='val')
print('Validation metrics:', val_metrics)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,12.925010,2.852019,0.421053,0.347734,0.349240
2,8.730747,1.698234,0.881579,0.861575,0.863301
3,4.941227,0.860706,0.979532,0.979120,0.979246
4,2.634437,0.423991,0.995614,0.995773,0.995607
5,1.533891,0.245053,0.997076,0.997023,0.997075
6,1.149013,0.198193,0.997076,0.997023,0.997075


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Training finished.
Best checkpoint: ..\model_training\outputs\koelectra_coupang_v1_1\checkpoint-965


early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Validation metrics: {'val_loss': 0.24379074573516846, 'val_accuracy': 0.9970760233918129, 'val_macro_f1': 0.9970225172135725, 'val_weighted_f1': 0.9970747537536852, 'val_runtime': 1.099, 'val_samples_per_second': 622.393, 'val_steps_per_second': 39.127, 'epoch': 6.0}


In [18]:
# 셀 7: 테스트셋 최종 평가
# test 지표를 계산하고 test_metrics.json으로 저장합니다.

def to_jsonable(value):
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, dict):
        return {k: to_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    return value

test_metrics = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix='test')
test_metrics = to_jsonable(test_metrics)

test_metrics_path = OUTPUT_DIR / 'test_metrics.json'
with open(test_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print('Test metrics:')
print(test_metrics)
print('Saved:', test_metrics_path)


early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Test metrics:
{'test_loss': 0.2499883621931076, 'test_accuracy': 0.9970760233918129, 'test_macro_f1': 0.9973234747755662, 'test_weighted_f1': 0.9970730778050758, 'test_runtime': 3.0218, 'test_samples_per_second': 565.897, 'test_steps_per_second': 35.41, 'epoch': 6.0}
Saved: ..\model_training\outputs\koelectra_coupang_v1_1\test_metrics.json


In [19]:
# 셀 8: 예측 결과/혼동행렬 저장
# test 예측 결과와 confidence, confusion matrix를 CSV로 저장합니다.

pred_output = trainer.predict(tokenized_test)
test_logits = pred_output.predictions
test_label_idx = pred_output.label_ids
test_pred_idx = np.argmax(test_logits, axis=-1)
test_confidence = torch.softmax(torch.tensor(test_logits), dim=-1).max(dim=-1).values.numpy()

test_pred_df = test_df.copy().reset_index(drop=True)
test_pred_df['true_class_idx'] = test_label_idx
test_pred_df['pred_class_idx'] = test_pred_idx
test_pred_df['true_label'] = [id2label[int(i)] for i in test_label_idx]
test_pred_df['pred_label'] = [id2label[int(i)] for i in test_pred_idx]
test_pred_df['confidence'] = test_confidence

predictions_path = OUTPUT_DIR / 'test_predictions.csv'
test_pred_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')

label_order = list(range(num_classes))
cm = confusion_matrix(test_label_idx, test_pred_idx, labels=label_order)
cm_df = pd.DataFrame(
    cm,
    index=[f'true_{id2label[i]}' for i in label_order],
    columns=[f'pred_{id2label[i]}' for i in label_order],
)
cm_path = OUTPUT_DIR / 'confusion_matrix.csv'
cm_df.to_csv(cm_path, encoding='utf-8-sig')

print('Saved:', predictions_path)
print('Saved:', cm_path)

report = classification_report(
    test_label_idx,
    test_pred_idx,
    labels=label_order,
    target_names=[id2label[i] for i in label_order],
    digits=4,
    zero_division=0,
)
print(report)


Saved: ..\model_training\outputs\koelectra_coupang_v1_1\test_predictions.csv
Saved: ..\model_training\outputs\koelectra_coupang_v1_1\confusion_matrix.csv
                           precision    recall  f1-score   support

                  unknown     1.0000    0.9667    0.9831        90
                go_hearbe     1.0000    1.0000    1.0000        60
          search_products     0.9836    1.0000    0.9917        60
      click_order_history     1.0000    1.0000    1.0000        60
               click_cart     1.0000    0.9833    0.9916        60
              click_login     1.0000    1.0000    1.0000        60
             click_logout     1.0000    1.0000    1.0000        60
read_current_page_actions     1.0000    1.0000    1.0000        60
      read_search_results     1.0000    1.0000    1.0000        60
     select_search_result     1.0000    1.0000    1.0000        60
        read_product_info     0.9836    1.0000    0.9917        60
    update_product_option     1.0000    1

In [20]:
# 셀 9: 샘플 추론
# 예시 문장 5개에 대한 Top-K 예측 라벨/점수를 출력합니다.

sample_texts = [
    '지금 장바구니에 담긴 상품을 읽어줘',
    '무선 이어폰 검색해줘',
    '결제할게 바로 진행해줘',
    '주문 내역 화면으로 이동해줘',
    '이 문장은 아무 기능 요청이 아니야',
]
top_k = 3

encoded = tokenizer(
    sample_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors='pt',
)

device = trainer.model.device
encoded = {k: v.to(device) for k, v in encoded.items()}

trainer.model.eval()
with torch.no_grad():
    logits = trainer.model(**encoded).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()

for text, prob in zip(sample_texts, probs):
    ranked_idx = prob.argsort()[::-1][:top_k]
    print(f'\n[text] {text}')
    for rank, idx in enumerate(ranked_idx, start=1):
        print(f'  {rank}. {id2label[int(idx)]} ({prob[idx]:.4f})')



[text] 지금 장바구니에 담긴 상품을 읽어줘
  1. read_cart_items (0.7761)
  2. click_cart (0.0418)
  3. read_reviews (0.0229)

[text] 무선 이어폰 검색해줘
  1. search_products (0.8402)
  2. read_product_info (0.0181)
  3. click_wishlist (0.0172)

[text] 결제할게 바로 진행해줘
  1. submit_payment (0.8395)
  2. click_buy_now (0.0342)
  3. request_exchange_return (0.0230)

[text] 주문 내역 화면으로 이동해줘
  1. click_order_history (0.7490)
  2. read_payment_order_info (0.0423)
  3. click_cart (0.0313)

[text] 이 문장은 아무 기능 요청이 아니야
  1. unknown (0.8787)
  2. unselect_cart_item (0.0289)
  3. click_logout (0.0126)


In [21]:
# 셀 10: 아티팩트 저장
# 모델/토크나이저를 저장하고 label_mapping.json을 기록합니다.

export_dir = OUTPUT_DIR / 'best_model_export'
trainer.save_model(str(export_dir))
tokenizer.save_pretrained(str(export_dir))

label_mapping_path = OUTPUT_DIR / 'label_mapping.json'
label_mapping_records = label_map_df[['label_id', 'label', 'class_idx']].to_dict(orient='records')
with open(label_mapping_path, 'w', encoding='utf-8') as f:
    json.dump(label_mapping_records, f, ensure_ascii=False, indent=2)

print('Saved model/tokenizer:', export_dir)
print('Saved label mapping:', label_mapping_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model/tokenizer: ..\model_training\outputs\koelectra_coupang_v1_1\best_model_export
Saved label mapping: ..\model_training\outputs\koelectra_coupang_v1_1\label_mapping.json


## CUDA OOM 발생 시 조정 순서

메모리 부족(CUDA OOM)이 나면 아래 순서로 조정하세요.
1. `TRAIN_BS: 8 -> 4`
2. `GRAD_ACCUM: 4 -> 8`
3. `MAX_LENGTH: 64 -> 48`
